In [ ]:
"""
================================================================================
Cross-Dataset Proteomics Analysis Pipeline
================================================================================
Analysis of peptide biomarkers across three independent datasets:
1. Cross-sectional cohort (HC, NC, MCI, PDD)
2. Longitudinal patient 149 (V1-V9)
3. Mouse model (NCE, NCL, DE)

Author: [Your Name]
Date: 2025-01-13
Version: 1.0 (Final)
================================================================================
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages
from scipy.stats import binomtest, pearsonr, ttest_ind
from scipy.optimize import curve_fit
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# Configuration
# ============================================================================

# Font settings
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 10

# Color palette
COLORS = {
    'Cross': {'main': '#E74C3C', 'dark': '#C0392B', 'light': '#FADBD8'},
    '149': {'main': '#3498DB', 'dark': '#2980B9', 'light': '#D4E6F1'},
    'Mouse': {'main': '#16A085', 'dark': '#1E8449', 'light': '#D5F5E3'},
    'NCE': '#27AE60',
    'NCL': '#F39C12',
    'DE': '#E74C3C',
    'Concordant': '#8E44AD',
    'Discordant': '#95A5A6'
}

# File paths
INPUT_FILE = "For data analysis total.xlsx"
OUTPUT_PREFIX = "PD_Proteomics"

print("="*80)
print("Cross-Dataset Proteomics Analysis Pipeline")
print("="*80)

# ============================================================================
# PART 1: Data Preprocessing
# ============================================================================

print("\n" + "="*80)
print("PART 1: Data Preprocessing")
print("="*80)

# Load data
print("\nLoading data...")
df_total = pd.read_excel(INPUT_FILE)
print(f"  Original data: {df_total.shape}")

# Set index
df_indexed = df_total.set_index('Sequence')

# Separate ENOLASE
enolase_row = df_indexed.loc['ENOLASE']
peptide_df = df_indexed.drop('ENOLASE')
print(f"  After ENOLASE separation: {peptide_df.shape}")

# Define sample groups
samples_149 = ['149V1', '149V3', '149V5', '149V7', '149V9']
samples_mouse = [col for col in df_indexed.columns
                if col.startswith(('NCE', 'NCL', 'DE')) and not col.startswith('DL')]
samples_hc = [col for col in df_indexed.columns if col.startswith('HC')]
samples_nc = [col for col in df_indexed.columns
             if col.startswith('NC') and not col.startswith(('NCE', 'NCL'))]
samples_mci = [col for col in df_indexed.columns if col.startswith('MCI')]
samples_pdd = [col for col in df_indexed.columns if col.startswith('PDD')]
samples_cross = samples_hc + samples_nc + samples_mci + samples_pdd

# Mouse subgroups
nce_samples = [s for s in samples_mouse if s.startswith('NCE')]
ncl_samples = [s for s in samples_mouse if s.startswith('NCL')]
de_samples = [s for s in samples_mouse if s.startswith('DE')]

print(f"\nSample groups:")
print(f"  149 Patient: {len(samples_149)} samples")
print(f"  Mouse: {len(samples_mouse)} samples (NCE:{len(nce_samples)}, NCL:{len(ncl_samples)}, DE:{len(de_samples)})")
print(f"  Cross-sectional: {len(samples_cross)} samples (HC:{len(samples_hc)}, NC:{len(samples_nc)}, MCI:{len(samples_mci)}, PDD:{len(samples_pdd)})")

# Split datasets
df_149 = peptide_df[samples_149].copy()
df_mouse = peptide_df[samples_mouse].copy()
df_cross = peptide_df[samples_cross].copy()

# Remove missing values
df_149_clean = df_149.dropna()
df_mouse_clean = df_mouse.dropna()
df_cross_clean = df_cross.dropna()

print(f"\nAfter missing value removal:")
print(f"  149: {len(df_149)} → {len(df_149_clean)} peptides")
print(f"  Mouse: {len(df_mouse)} → {len(df_mouse_clean)} peptides")
print(f"  Cross: {len(df_cross)} → {len(df_cross_clean)} peptides")

# ENOLASE normalization
print("\nENOLASE normalization...")
df_149_enolase = df_149_clean.copy()
for sample in samples_149:
    df_149_enolase[sample] = df_149_clean[sample] / enolase_row[sample]

df_mouse_enolase = df_mouse_clean.copy()
for sample in samples_mouse:
    df_mouse_enolase[sample] = df_mouse_clean[sample] / enolase_row[sample]

df_cross_enolase = df_cross_clean.copy()
for sample in samples_cross:
    df_cross_enolase[sample] = df_cross_clean[sample] / enolase_row[sample]

# Row median normalization
print("Row median normalization...")
row_medians_149 = df_149_enolase.median(axis=1)
df_149_median = df_149_enolase.div(row_medians_149, axis=0)

row_medians_mouse = df_mouse_enolase.median(axis=1)
df_mouse_median = df_mouse_enolase.div(row_medians_mouse, axis=0)

row_medians_cross = df_cross_enolase.median(axis=1)
df_cross_median = df_cross_enolase.div(row_medians_cross, axis=0)

# Log2 transformation
print("Log2 transformation...")
pseudocount = 1e-8
df_149_log2 = np.log2(df_149_median + pseudocount)
df_mouse_log2 = np.log2(df_mouse_median + pseudocount)
df_cross_log2 = np.log2(df_cross_median + pseudocount)

# Z-score normalization
print("Z-score normalization...")
def zscore_row(df):
    row_mean = df.mean(axis=1)
    row_std = df.std(axis=1)
    return df.sub(row_mean, axis=0).div(row_std, axis=0)

df_149_zscore = zscore_row(df_149_log2)
df_mouse_zscore = zscore_row(df_mouse_log2)
df_cross_zscore = zscore_row(df_cross_log2)

# Find common peptides
peptides_149 = set(df_149_log2.index)
peptides_mouse = set(df_mouse_log2.index)
peptides_cross = set(df_cross_log2.index)

common_all = peptides_149 & peptides_mouse & peptides_cross
common_149_mouse = peptides_149 & peptides_mouse

common_all_list = sorted(list(common_all))
common_149_mouse_list = sorted(list(common_149_mouse))

print(f"\nCommon peptides:")
print(f"  All datasets: {len(common_all)} peptides")
print(f"  149+Mouse: {len(common_149_mouse)} peptides")

# Save preprocessed data
with pd.ExcelWriter(f'{OUTPUT_PREFIX}_01_Preprocessed_Data.xlsx', engine='openpyxl') as writer:
    df_149_log2.to_excel(writer, sheet_name='149_Log2')
    df_mouse_log2.to_excel(writer, sheet_name='Mouse_Log2')
    df_cross_log2.to_excel(writer, sheet_name='Cross_Log2')
    df_149_zscore.to_excel(writer, sheet_name='149_Zscore')
    df_mouse_zscore.to_excel(writer, sheet_name='Mouse_Zscore')
    df_cross_zscore.to_excel(writer, sheet_name='Cross_Zscore')
    pd.DataFrame({'Common_All': common_all_list}).to_excel(writer, sheet_name='Common_Peptides', index=False)

print(f"\n✅ Preprocessing complete!")
print(f"   Saved: {OUTPUT_PREFIX}_01_Preprocessed_Data.xlsx")

# ============================================================================
# PART 2: Heatmap Visualization
# ============================================================================

print("\n" + "="*80)
print("PART 2: Heatmap Visualization")
print("="*80)

# Order mouse samples
samples_mouse_ordered = nce_samples + ncl_samples + de_samples

with PdfPages(f'{OUTPUT_PREFIX}_02_Heatmaps.pdf') as pdf:

    # Page 1: All peptides - Individual samples
    print("\nPage 1: All peptides - Individual samples...")
    fig, axes = plt.subplots(1, 3, figsize=(20, 12))

    # 149
    ax = axes[0]
    sns.heatmap(df_149_zscore, cmap='RdBu_r', center=0, vmin=-2, vmax=2,
                cbar_kws={'label': 'Z-score'}, ax=ax,
                xticklabels=['V1\n(NC)', 'V3\n(MCI)', 'V5\n(MCI)', 'V7\n(PDD)', 'V9\n(PDD)'],
                yticklabels=False)
    ax.set_title(f'149 Patient\n(n={len(df_149_zscore)} peptides)', fontsize=12, fontweight='bold')

    # Mouse
    ax = axes[1]
    df_mouse_zscore_ordered = df_mouse_zscore[samples_mouse_ordered]
    sns.heatmap(df_mouse_zscore_ordered, cmap='RdBu_r', center=0, vmin=-2, vmax=2,
                cbar_kws={'label': 'Z-score'}, ax=ax, yticklabels=False)
    ax.set_title(f'Mouse\n(n={len(df_mouse_zscore)} peptides)', fontsize=12, fontweight='bold')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90, ha='right')

    # Cross
    ax = axes[2]
    sns.heatmap(df_cross_zscore, cmap='RdBu_r', center=0, vmin=-2, vmax=2,
                cbar_kws={'label': 'Z-score'}, ax=ax, yticklabels=False)
    ax.set_title(f'Cross-sectional\n(n={len(df_cross_zscore)} peptides)', fontsize=12, fontweight='bold')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90, ha='right', fontsize=6)

    plt.suptitle('Heatmap: All Peptides - Individual Samples', fontsize=14, fontweight='bold')
    plt.tight_layout()
    pdf.savefig(fig, bbox_inches='tight')
    plt.close()

    # Page 2: Common peptides - Individual samples
    print("Page 2: Common peptides - Individual samples...")
    fig, axes = plt.subplots(1, 3, figsize=(20, 10))

    df_149_common = df_149_zscore.loc[common_all_list]
    df_mouse_common = df_mouse_zscore.loc[common_all_list]
    df_cross_common = df_cross_zscore.loc[common_all_list]

    # 149
    ax = axes[0]
    sns.heatmap(df_149_common, cmap='RdBu_r', center=0, vmin=-2, vmax=2,
                cbar_kws={'label': 'Z-score'}, ax=ax,
                xticklabels=['V1\n(NC)', 'V3\n(MCI)', 'V5\n(MCI)', 'V7\n(PDD)', 'V9\n(PDD)'],
                yticklabels=True)
    ax.set_title(f'149 Patient - Common\n(n={len(common_all_list)} peptides)', fontsize=12, fontweight='bold')

    # Mouse
    ax = axes[1]
    df_mouse_common_ordered = df_mouse_common[samples_mouse_ordered]
    sns.heatmap(df_mouse_common_ordered, cmap='RdBu_r', center=0, vmin=-2, vmax=2,
                cbar_kws={'label': 'Z-score'}, ax=ax, yticklabels=True)
    ax.set_title(f'Mouse - Common\n(n={len(common_all_list)} peptides)', fontsize=12, fontweight='bold')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90, ha='right')

    # Cross
    ax = axes[2]
    sns.heatmap(df_cross_common, cmap='RdBu_r', center=0, vmin=-2, vmax=2,
                cbar_kws={'label': 'Z-score'}, ax=ax, yticklabels=True)
    ax.set_title(f'Cross-sectional - Common\n(n={len(common_all_list)} peptides)', fontsize=12, fontweight='bold')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90, ha='right', fontsize=6)

    plt.suptitle('Heatmap: Common Peptides - Individual Samples', fontsize=14, fontweight='bold')
    plt.tight_layout()
    pdf.savefig(fig, bbox_inches='tight')
    plt.close()

    # Page 3: Group averages
    print("Page 3: Group averages...")
    fig, axes = plt.subplots(1, 3, figsize=(18, 12))

    # 149
    ax = axes[0]
    sns.heatmap(df_149_zscore, cmap='RdBu_r', center=0, vmin=-2, vmax=2,
                cbar_kws={'label': 'Z-score'}, ax=ax,
                xticklabels=['V1\n(NC)', 'V3\n(MCI)', 'V5\n(MCI)', 'V7\n(PDD)', 'V9\n(PDD)'],
                yticklabels=False)
    ax.set_title(f'149 Patient\n(n={len(df_149_zscore)} peptides)', fontsize=12, fontweight='bold')

    # Mouse - group average
    ax = axes[1]
    mouse_grouped = pd.DataFrame({
        'NCE': df_mouse_zscore[nce_samples].mean(axis=1),
        'NCL': df_mouse_zscore[ncl_samples].mean(axis=1),
        'DE': df_mouse_zscore[de_samples].mean(axis=1)
    })
    sns.heatmap(mouse_grouped, cmap='RdBu_r', center=0, vmin=-2, vmax=2,
                cbar_kws={'label': 'Z-score'}, ax=ax, yticklabels=False)
    ax.set_title(f'Mouse - Group Average\n(n={len(mouse_grouped)} peptides)', fontsize=12, fontweight='bold')

    # Cross - group average
    ax = axes[2]
    cross_grouped = pd.DataFrame({
        'HC': df_cross_zscore[samples_hc].mean(axis=1),
        'NC': df_cross_zscore[samples_nc].mean(axis=1),
        'MCI': df_cross_zscore[samples_mci].mean(axis=1),
        'PDD': df_cross_zscore[samples_pdd].mean(axis=1)
    })
    sns.heatmap(cross_grouped, cmap='RdBu_r', center=0, vmin=-2, vmax=2,
                cbar_kws={'label': 'Z-score'}, ax=ax, yticklabels=False)
    ax.set_title(f'Cross-sectional - Group Average\n(n={len(cross_grouped)} peptides)', fontsize=12, fontweight='bold')

    plt.suptitle('Heatmap: Group Averages', fontsize=14, fontweight='bold')
    plt.tight_layout()
    pdf.savefig(fig, bbox_inches='tight')
    plt.close()

    # Page 4-6: Stage differences
    print("Page 4-6: Stage differences...")

    # 149 differences
    fig, ax = plt.subplots(figsize=(8, 10))
    df_149_diff = pd.DataFrame({
        'V3-V1': df_149_common['149V3'] - df_149_common['149V1'],
        'V5-V3': df_149_common['149V5'] - df_149_common['149V3'],
        'V7-V5': df_149_common['149V7'] - df_149_common['149V5'],
        'V9-V7': df_149_common['149V9'] - df_149_common['149V7']
    })
    sns.heatmap(df_149_diff, cmap='RdBu_r', center=0, vmin=-3, vmax=3,
                cbar_kws={'label': 'Z-score Difference'}, ax=ax, yticklabels=True)
    ax.set_title(f'149 Patient: Stage Differences\n(n={len(common_all_list)} peptides)',
                fontsize=12, fontweight='bold')
    plt.tight_layout()
    pdf.savefig(fig, bbox_inches='tight')
    plt.close()

    # Mouse differences
    fig, ax = plt.subplots(figsize=(7, 10))
    mouse_common_grouped = pd.DataFrame({
        'NCE': df_mouse_common[nce_samples].mean(axis=1),
        'NCL': df_mouse_common[ncl_samples].mean(axis=1),
        'DE': df_mouse_common[de_samples].mean(axis=1)
    })
    df_mouse_diff = pd.DataFrame({
        'NCL-NCE': mouse_common_grouped['NCL'] - mouse_common_grouped['NCE'],
        'DE-NCL': mouse_common_grouped['DE'] - mouse_common_grouped['NCL'],
        'DE-NCE': mouse_common_grouped['DE'] - mouse_common_grouped['NCE']
    })
    sns.heatmap(df_mouse_diff, cmap='RdBu_r', center=0, vmin=-3, vmax=3,
                cbar_kws={'label': 'Z-score Difference'}, ax=ax, yticklabels=True)
    ax.set_title(f'Mouse: Group Differences\n(n={len(common_all_list)} peptides)',
                fontsize=12, fontweight='bold')
    plt.tight_layout()
    pdf.savefig(fig, bbox_inches='tight')
    plt.close()

    # Cross differences
    fig, ax = plt.subplots(figsize=(9, 10))
    cross_common_grouped = pd.DataFrame({
        'HC': df_cross_common[samples_hc].mean(axis=1),
        'NC': df_cross_common[samples_nc].mean(axis=1),
        'MCI': df_cross_common[samples_mci].mean(axis=1),
        'PDD': df_cross_common[samples_pdd].mean(axis=1)
    })
    df_cross_diff = pd.DataFrame({
        'NC-HC': cross_common_grouped['NC'] - cross_common_grouped['HC'],
        'MCI-NC': cross_common_grouped['MCI'] - cross_common_grouped['NC'],
        'PDD-MCI': cross_common_grouped['PDD'] - cross_common_grouped['MCI'],
        'PDD-HC': cross_common_grouped['PDD'] - cross_common_grouped['HC']
    })
    sns.heatmap(df_cross_diff, cmap='RdBu_r', center=0, vmin=-3, vmax=3,
                cbar_kws={'label': 'Z-score Difference'}, ax=ax, yticklabels=True)
    ax.set_title(f'Cross-sectional: Stage Differences\n(n={len(common_all_list)} peptides)',
                fontsize=12, fontweight='bold')
    plt.tight_layout()
    pdf.savefig(fig, bbox_inches='tight')
    plt.close()

print(f"\n✅ Heatmaps complete!")
print(f"   Saved: {OUTPUT_PREFIX}_02_Heatmaps.pdf")

# ============================================================================
# PART 3: Pattern Analysis (Monotonicity-based)
# ============================================================================

print("\n" + "="*80)
print("PART 3: Pattern Analysis (Monotonicity)")
print("="*80)

# Prepare pattern data
cross_pattern = pd.DataFrame({
    'NC': cross_common_grouped['NC'],
    'MCI': cross_common_grouped['MCI'],
    'PDD': cross_common_grouped['PDD']
}, index=common_all_list)

p149_pattern = pd.DataFrame({
    'NC': df_149_common['149V1'],
    'MCI': (df_149_common['149V3'] + df_149_common['149V5']) / 2,
    'PDD': (df_149_common['149V7'] + df_149_common['149V9']) / 2
}, index=common_all_list)

mouse_pattern = mouse_common_grouped[['NCE', 'NCL', 'DE']].copy()

# Classify monotonicity
def classify_monotonicity_3stage(row, threshold=0.0):
    """Classify monotonic pattern"""
    v1, v2, v3 = row.values
    if (v2 > v1 + threshold) and (v3 > v2 + threshold):
        return '↑'
    elif (v2 < v1 - threshold) and (v3 < v2 - threshold):
        return '↓'
    else:
        return 'Non-monotonic'

cross_direction = cross_pattern.apply(classify_monotonicity_3stage, axis=1, threshold=0.0)
p149_direction = p149_pattern.apply(classify_monotonicity_3stage, axis=1, threshold=0.0)
mouse_direction = mouse_pattern.apply(classify_monotonicity_3stage, axis=1, threshold=0.0)

# Create direction dataframe
direction_df = pd.DataFrame({
    'Cross': cross_direction,
    '149': p149_direction,
    'Mouse': mouse_direction
}, index=common_all_list)

# Check concordance
def check_concordance(row):
    if row['Cross'] == row['149'] == row['Mouse']:
        if row['Cross'] == '↑':
            return 'Concordant ↑'
        elif row['Cross'] == '↓':
            return 'Concordant ↓'
        else:
            return 'Concordant Non-mono'
    else:
        return 'Discordant'

direction_df['Pattern'] = direction_df.apply(check_concordance, axis=1)

# Statistics
pattern_counts = direction_df['Pattern'].value_counts()
n_total = len(common_all_list)
n_concordant_up = pattern_counts.get('Concordant ↑', 0)
n_concordant_down = pattern_counts.get('Concordant ↓', 0)
n_concordant_total = n_concordant_up + n_concordant_down
expected_prob = 2/27  # ~7.4%

binom_result = binomtest(n_concordant_total, n_total, expected_prob, alternative='greater')
p_value = binom_result.pvalue

print(f"\nPattern Summary:")
for pattern, count in pattern_counts.items():
    print(f"  {pattern}: {count} ({100*count/n_total:.1f}%)")

print(f"\nStatistics:")
print(f"  Total: {n_total} peptides")
print(f"  Concordant (↑+↓): {n_concordant_total} ({100*n_concordant_total/n_total:.1f}%)")
print(f"  Expected: ~{expected_prob*100:.1f}%")
print(f"  p-value: {p_value:.4f}")

# Get concordant peptides
concordant_up = direction_df[direction_df['Pattern'] == 'Concordant ↑'].index.tolist()
concordant_down = direction_df[direction_df['Pattern'] == 'Concordant ↓'].index.tolist()

print(f"\nConcordant ↑ peptides (n={len(concordant_up)}):")
for i, pep in enumerate(concordant_up, 1):
    print(f"  {i}. {pep}")

# Visualization
with PdfPages(f'{OUTPUT_PREFIX}_03_Pattern_Analysis.pdf') as pdf:

    # Main figure
    print("\nGenerating pattern analysis figure...")
    fig = plt.figure(figsize=(18, 12))

    # A) Dot Plot
    ax_dot = fig.add_axes([0.05, 0.35, 0.50, 0.60])

    datasets = ['Cross', '149', 'Mouse']
    sorted_peptides = direction_df.sort_values('Pattern').index.tolist()
    n_peptides = len(sorted_peptides)

    for i, pep in enumerate(sorted_peptides):
        for j, ds in enumerate(datasets):
            direction = direction_df.loc[pep, ds]
            if direction == '↑':
                color = '#E74C3C'
                marker = '^'
            elif direction == '↓':
                color = '#3498DB'
                marker = 'v'
            else:
                color = '#95A5A6'
                marker = 'o'

            ax_dot.scatter(j, i, c=color, s=150, marker=marker,
                          edgecolors='black', linewidths=0.5, zorder=3)

    # Concordant backgrounds
    y_positions = {pep: i for i, pep in enumerate(sorted_peptides)}

    if concordant_up:
        y_min = min([y_positions[p] for p in concordant_up]) - 0.5
        y_max = max([y_positions[p] for p in concordant_up]) + 0.5
        ax_dot.axhspan(y_min, y_max, alpha=0.15, color='red', zorder=1)
        ax_dot.text(3.2, (y_min + y_max) / 2, f'Concordant ↑\n({len(concordant_up)})',
                   va='center', fontsize=10, color='#E74C3C', fontweight='bold')

    if concordant_down:
        y_min = min([y_positions[p] for p in concordant_down]) - 0.5
        y_max = max([y_positions[p] for p in concordant_down]) + 0.5
        ax_dot.axhspan(y_min, y_max, alpha=0.15, color='blue', zorder=1)
        ax_dot.text(3.2, (y_min + y_max) / 2, f'Concordant ↓\n({len(concordant_down)})',
                   va='center', fontsize=10, color='#3498DB', fontweight='bold')

    ax_dot.set_xticks(range(len(datasets)))
    ax_dot.set_xticklabels(['Cross-sectional\n(NC→MCI→PDD)',
                            '149 Patient\n(V1→MCI→PDD)',
                            'Mouse\n(NCE→NCL→DE)'], fontsize=11)
    ax_dot.set_yticks(range(n_peptides))
    ax_dot.set_yticklabels([p[:25]+'...' if len(p)>25 else p for p in sorted_peptides], fontsize=7)
    ax_dot.set_xlim(-0.5, 3.5)
    ax_dot.set_ylim(-0.5, n_peptides - 0.5)
    ax_dot.set_title('A) Monotonic Pattern Across Datasets', fontsize=13, fontweight='bold')
    ax_dot.grid(True, alpha=0.3, axis='x')

    legend_elements = [
        plt.scatter([], [], c='#E74C3C', s=100, marker='^', edgecolors='black', label='Monotonic ↑'),
        plt.scatter([], [], c='#3498DB', s=100, marker='v', edgecolors='black', label='Monotonic ↓'),
        plt.scatter([], [], c='#95A5A6', s=100, marker='o', edgecolors='black', label='Non-monotonic')
    ]
    ax_dot.legend(handles=legend_elements, loc='upper left', fontsize=10)

    # B) Summary pie chart
    ax_summary = fig.add_axes([0.62, 0.60, 0.33, 0.33])

    colors_pie = [COLORS['Concordant'], COLORS['Discordant']]
    wedges, texts, autotexts = ax_summary.pie(
        [n_concordant_total, n_total - n_concordant_total],
        labels=[f'Concordant\n({n_concordant_total})', f'Discordant\n({n_total - n_concordant_total})'],
        colors=colors_pie,
        autopct='%1.1f%%',
        startangle=90,
        explode=(0.05, 0),
        textprops={'fontsize': 11, 'fontweight': 'bold'}
    )
    ax_summary.set_title('B) Pattern Concordance', fontsize=13, fontweight='bold')

    stat_text = f"Expected: ~7.4%\nObserved: {n_concordant_total/n_total*100:.1f}%\np = {p_value:.4f}"
    if p_value < 0.001:
        stat_text = f"Expected: ~7.4%\nObserved: {n_concordant_total/n_total*100:.1f}%\np < 0.001 ***"
    ax_summary.text(0, -1.5, stat_text, ha='center', va='top', fontsize=11,
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))

    # C) Line plots
    ax_line1 = fig.add_axes([0.08, 0.05, 0.25, 0.22])
    ax_line2 = fig.add_axes([0.40, 0.05, 0.25, 0.22])
    ax_line3 = fig.add_axes([0.72, 0.05, 0.25, 0.22])

    x_stages = [0, 1, 2]
    stage_labels = ['NC', 'MCI', 'PDD']

    # C-1) Concordant ↑
    if concordant_up:
        for pep in concordant_up:
            ax_line1.plot(x_stages, cross_pattern.loc[pep].values, '-',
                         color=COLORS['Cross']['main'], alpha=0.2, linewidth=1)
            ax_line1.plot(x_stages, p149_pattern.loc[pep].values, '-',
                         color=COLORS['149']['main'], alpha=0.2, linewidth=1)
            ax_line1.plot(x_stages, mouse_pattern.loc[pep].values, '-',
                         color=COLORS['Mouse']['main'], alpha=0.2, linewidth=1)

        mean_cross = cross_pattern.loc[concordant_up].mean()
        mean_149 = p149_pattern.loc[concordant_up].mean()
        mean_mouse = mouse_pattern.loc[concordant_up].mean()

        ax_line1.plot(x_stages, mean_cross.values, 'o-', color=COLORS['Cross']['dark'],
                     linewidth=3, markersize=10, label='Cross', markeredgecolor='black', markeredgewidth=1.5)
        ax_line1.plot(x_stages, mean_149.values, 's-', color=COLORS['149']['dark'],
                     linewidth=3, markersize=10, label='149', markeredgecolor='black', markeredgewidth=1.5)
        ax_line1.plot(x_stages, mean_mouse.values, '^-', color=COLORS['Mouse']['dark'],
                     linewidth=3, markersize=10, label='Mouse', markeredgecolor='black', markeredgewidth=1.5)

    ax_line1.axhline(y=0, color='gray', linestyle='-', alpha=0.5)
    ax_line1.set_xticks(x_stages)
    ax_line1.set_xticklabels(stage_labels, fontsize=10)
    ax_line1.set_xlabel('Disease Stage', fontsize=10)
    ax_line1.set_ylabel('Z-score', fontsize=10)
    ax_line1.set_title(f'C-1) Concordant ↑ (n={len(concordant_up)})',
                      fontsize=11, fontweight='bold', color='#E74C3C')
    ax_line1.legend(fontsize=9, loc='upper left')
    ax_line1.grid(True, alpha=0.3)

    # C-2) Concordant ↓
    if concordant_down:
        for pep in concordant_down:
            ax_line2.plot(x_stages, cross_pattern.loc[pep].values, '-',
                         color=COLORS['Cross']['main'], alpha=0.2, linewidth=1)
            ax_line2.plot(x_stages, p149_pattern.loc[pep].values, '-',
                         color=COLORS['149']['main'], alpha=0.2, linewidth=1)
            ax_line2.plot(x_stages, mouse_pattern.loc[pep].values, '-',
                         color=COLORS['Mouse']['main'], alpha=0.2, linewidth=1)

        mean_cross = cross_pattern.loc[concordant_down].mean()
        mean_149 = p149_pattern.loc[concordant_down].mean()
        mean_mouse = mouse_pattern.loc[concordant_down].mean()

        ax_line2.plot(x_stages, mean_cross.values, 'o-', color=COLORS['Cross']['dark'],
                     linewidth=3, markersize=10, label='Cross', markeredgecolor='black', markeredgewidth=1.5)
        ax_line2.plot(x_stages, mean_149.values, 's-', color=COLORS['149']['dark'],
                     linewidth=3, markersize=10, label='149', markeredgecolor='black', markeredgewidth=1.5)
        ax_line2.plot(x_stages, mean_mouse.values, '^-', color=COLORS['Mouse']['dark'],
                     linewidth=3, markersize=10, label='Mouse', markeredgecolor='black', markeredgewidth=1.5)
        ax_line2.legend(fontsize=9, loc='upper right')
    else:
        ax_line2.text(0.5, 0.5, 'No Concordant ↓\npeptides', transform=ax_line2.transAxes,
                     ha='center', va='center', fontsize=12, color='gray')

    ax_line2.axhline(y=0, color='gray', linestyle='-', alpha=0.5)
    ax_line2.set_xticks(x_stages)
    ax_line2.set_xticklabels(stage_labels, fontsize=10)
    ax_line2.set_xlabel('Disease Stage', fontsize=10)
    ax_line2.set_ylabel('Z-score', fontsize=10)
    ax_line2.set_title(f'C-2) Concordant ↓ (n={len(concordant_down)})',
                      fontsize=11, fontweight='bold', color='#3498DB')
    ax_line2.grid(True, alpha=0.3)

    # C-3) Discordant
    discordant_peps = direction_df[direction_df['Pattern'] == 'Discordant'].index.tolist()

    if discordant_peps:
        for pep in discordant_peps:
            ax_line3.plot(x_stages, cross_pattern.loc[pep].values, '-',
                         color=COLORS['Cross']['main'], alpha=0.3, linewidth=1)
            ax_line3.plot(x_stages, p149_pattern.loc[pep].values, '-',
                         color=COLORS['149']['main'], alpha=0.3, linewidth=1)
            ax_line3.plot(x_stages, mouse_pattern.loc[pep].values, '-',
                         color=COLORS['Mouse']['main'], alpha=0.3, linewidth=1)

        if len(discordant_peps) > 0:
            mean_cross = cross_pattern.loc[discordant_peps].mean()
            mean_149 = p149_pattern.loc[discordant_peps].mean()
            mean_mouse = mouse_pattern.loc[discordant_peps].mean()

            ax_line3.plot(x_stages, mean_cross.values, 'o-', color=COLORS['Cross']['dark'],
                         linewidth=3, markersize=10, label='Cross', markeredgecolor='black', markeredgewidth=1.5)
            ax_line3.plot(x_stages, mean_149.values, 's-', color=COLORS['149']['dark'],
                         linewidth=3, markersize=10, label='149', markeredgecolor='black', markeredgewidth=1.5)
            ax_line3.plot(x_stages, mean_mouse.values, '^-', color=COLORS['Mouse']['dark'],
                         linewidth=3, markersize=10, label='Mouse', markeredgecolor='black', markeredgewidth=1.5)
            ax_line3.legend(fontsize=9)

    ax_line3.axhline(y=0, color='gray', linestyle='-', alpha=0.5)
    ax_line3.set_xticks(x_stages)
    ax_line3.set_xticklabels(stage_labels, fontsize=10)
    ax_line3.set_xlabel('Disease Stage', fontsize=10)
    ax_line3.set_ylabel('Z-score', fontsize=10)
    ax_line3.set_title(f'C-3) Discordant (n={len(discordant_peps)})',
                      fontsize=11, fontweight='bold', color='#7F8C8D')
    ax_line3.grid(True, alpha=0.3)

    fig.suptitle(f'Cross-Dataset Monotonic Pattern Analysis\n(n={n_total} common peptides)',
                fontsize=15, fontweight='bold', y=0.98)

    pdf.savefig(fig, bbox_inches='tight', dpi=300)
    plt.close()

    # Data table page
    print("Generating pattern table...")
    fig, ax = plt.subplots(figsize=(14, 10))
    ax.axis('off')

    table_data = []
    for pep in sorted_peptides:
        row = [
            pep[:35] + '...' if len(pep) > 35 else pep,
            direction_df.loc[pep, 'Cross'],
            direction_df.loc[pep, '149'],
            direction_df.loc[pep, 'Mouse'],
            direction_df.loc[pep, 'Pattern']
        ]
        table_data.append(row)

    table = ax.table(
        cellText=table_data,
        colLabels=['Peptide', 'Cross\n(NC<MCI<PDD)', '149\n(V1<MCI<PDD)', 'Mouse\n(NCE<NCL<DE)', 'Pattern'],
        loc='center',
        cellLoc='center'
    )

    table.auto_set_font_size(False)
    table.set_fontsize(7)
    table.scale(1.2, 1.5)

    for i in range(5):
        table[(0, i)].set_facecolor('#34495E')
        table[(0, i)].set_text_props(color='white', fontweight='bold')

    for row_idx, pep in enumerate(sorted_peptides, start=1):
        pattern = direction_df.loc[pep, 'Pattern']
        if 'Concordant ↑' in pattern:
            for col in range(5):
                table[(row_idx, col)].set_facecolor('#FADBD8')
        elif 'Concordant ↓' in pattern:
            for col in range(5):
                table[(row_idx, col)].set_facecolor('#D4E6F1')

    ax.set_title('Monotonic Pattern Classification Table', fontsize=14, fontweight='bold', pad=20)

    legend_text = "Criteria:\n↑ = Monotonic Increase (Stage1 < Stage2 < Stage3)\n↓ = Monotonic Decrease (Stage1 > Stage2 > Stage3)\nNon-monotonic = Direction change between stages"
    ax.text(0.5, 0.02, legend_text, ha='center', va='bottom', transform=ax.transAxes,
           fontsize=9, bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

    pdf.savefig(fig, bbox_inches='tight')
    plt.close()

# Save pattern results
with pd.ExcelWriter(f'{OUTPUT_PREFIX}_03_Pattern_Results.xlsx', engine='openpyxl') as writer:
    direction_df.to_excel(writer, sheet_name='Direction_Classification')
    cross_pattern.to_excel(writer, sheet_name='Cross_Values')
    p149_pattern.to_excel(writer, sheet_name='149_Values')
    mouse_pattern.to_excel(writer, sheet_name='Mouse_Values')

    summary_stats = pd.DataFrame({
        'Metric': ['Total Peptides', 'Concordant ↑', 'Concordant ↓',
                   'Total Concordant', 'Discordant',
                   'Concordance Rate (%)', 'Expected (%)', 'p-value'],
        'Value': [n_total, n_concordant_up, n_concordant_down,
                 n_concordant_total, pattern_counts.get('Discordant', 0),
                 f"{n_concordant_total/n_total*100:.1f}",
                 f"{expected_prob*100:.1f}",
                 f"{p_value:.4f}"]
    })
    summary_stats.to_excel(writer, sheet_name='Summary', index=False)

print(f"\n✅ Pattern analysis complete!")
print(f"   Saved: {OUTPUT_PREFIX}_03_Pattern_Analysis.pdf")
print(f"   Saved: {OUTPUT_PREFIX}_03_Pattern_Results.xlsx")

# ============================================================================
# PART 4: Anchored Trajectory Analysis (Concordant peptides only)
# ============================================================================

if len(concordant_up) > 0:
    print("\n" + "="*80)
    print("PART 4: Anchored Trajectory Analysis")
    print("="*80)

    print(f"\nUsing {len(concordant_up)} concordant ↑ peptides")

    # Anchored Min-Max normalization
    def anchored_minmax_normalize(df, peptide_list, start_col, end_col):
        df_norm = pd.DataFrame(index=peptide_list, columns=df.columns)

        for pep in peptide_list:
            start_val = df.loc[pep, start_col]
            end_val = df.loc[pep, end_col]
            range_val = end_val - start_val

            if abs(range_val) > 1e-6:
                for col in df.columns:
                    df_norm.loc[pep, col] = (df.loc[pep, col] - start_val) / range_val
            else:
                for col in df.columns:
                    df_norm.loc[pep, col] = 0.5

        return df_norm.astype(float)

    # 149: V1=0, V9=1
    df_149_log2_concordant = df_149_log2.loc[concordant_up]
    df_149_anchored = anchored_minmax_normalize(
        df_149_log2_concordant,
        concordant_up,
        '149V1',
        '149V9'
    )

    # Mouse: NCE=0, DE=1
    df_mouse_log2_concordant = df_mouse_log2.loc[concordant_up]
    mouse_grouped_concordant = pd.DataFrame({
        'NCE': df_mouse_log2_concordant[nce_samples].mean(axis=1),
        'NCL': df_mouse_log2_concordant[ncl_samples].mean(axis=1),
        'DE': df_mouse_log2_concordant[de_samples].mean(axis=1)
    }, index=concordant_up)

    df_mouse_anchored = anchored_minmax_normalize(
        mouse_grouped_concordant,
        concordant_up,
        'NCE',
        'DE'
    )

    print(f"\n149 Anchored (mean):")
    for col in df_149_anchored.columns:
        print(f"  {col}: {df_149_anchored[col].mean():.3f}")

    print(f"\nMouse Anchored (mean):")
    for col in df_mouse_anchored.columns:
        print(f"  {col}: {df_mouse_anchored[col].mean():.3f}")

    # Trajectory model
    visit_to_time = {'149V1': 1, '149V3': 3, '149V5': 5, '149V7': 7, '149V9': 9}
    means_149 = df_149_anchored.mean(axis=0)
    human_timepoints = np.array([visit_to_time[v] for v in df_149_anchored.columns])

    # Model functions
    def linear_func(x, a, b):
        return a * x + b

    def poly2_func(x, a, b, c):
        return a * x**2 + b * x + c

    def poly3_func(x, a, b, c, d):
        return a * x**3 + b * x**2 + c * x + d

    # Fit models
    models = {}

    try:
        params_linear, _ = curve_fit(linear_func, human_timepoints, means_149)
        y_pred_linear = linear_func(human_timepoints, *params_linear)
        r2_linear = 1 - np.sum((means_149 - y_pred_linear)**2) / np.sum((means_149 - means_149.mean())**2)
        models['Linear'] = {'func': linear_func, 'params': params_linear, 'r2': r2_linear}
    except: pass

    try:
        params_poly2, _ = curve_fit(poly2_func, human_timepoints, means_149)
        y_pred_poly2 = poly2_func(human_timepoints, *params_poly2)
        r2_poly2 = 1 - np.sum((means_149 - y_pred_poly2)**2) / np.sum((means_149 - means_149.mean())**2)
        models['Polynomial-2'] = {'func': poly2_func, 'params': params_poly2, 'r2': r2_poly2}
    except: pass

    try:
        params_poly3, _ = curve_fit(poly3_func, human_timepoints, means_149)
        y_pred_poly3 = poly3_func(human_timepoints, *params_poly3)
        r2_poly3 = 1 - np.sum((means_149 - y_pred_poly3)**2) / np.sum((means_149 - means_149.mean())**2)
        models['Polynomial-3'] = {'func': poly3_func, 'params': params_poly3, 'r2': r2_poly3}
    except: pass

    best_model_name = max(models.keys(), key=lambda k: models[k]['r2'])
    best_model = models[best_model_name]

    print(f"\nTrajectory Models:")
    for name, model in models.items():
        print(f"  {name}: R² = {model['r2']:.4f}")
    print(f"  → Selected: {best_model_name}")

    # NCL projection
    def find_timepoint_for_value(func, params, target_value, time_range=(1, 9)):
        t_test = np.linspace(time_range[0], time_range[1], 1000)
        y_test = func(t_test, *params)
        idx = np.argmin(np.abs(y_test - target_value))
        return t_test[idx]

    nce_mean = df_mouse_anchored['NCE'].mean()
    ncl_mean = df_mouse_anchored['NCL'].mean()
    de_mean = df_mouse_anchored['DE'].mean()

    ncl_projected_time = find_timepoint_for_value(
        best_model['func'],
        best_model['params'],
        ncl_mean
    )

    print(f"\nNCL Projection:")
    print(f"  NCL value: {ncl_mean:.3f} (0-1 scale)")
    print(f"  Projected time: V{ncl_projected_time:.2f}")

    # Individual peptide positions
    ncl_positions_dict = {}
    for pep in concordant_up:
        ncl_val = df_mouse_anchored.loc[pep, 'NCL']
        ncl_time = find_timepoint_for_value(best_model['func'], best_model['params'], ncl_val)
        ncl_positions_dict[pep] = {'value': ncl_val, 'time': ncl_time}

    # Visualization
    with PdfPages(f'{OUTPUT_PREFIX}_04_Trajectory_Analysis.pdf') as pdf:

        # Main trajectory figure
        print("\nGenerating trajectory figures...")
        fig, ax = plt.subplots(figsize=(14, 8))

        # Individual peptides (gray)
        for pep in concordant_up:
            y_vals = df_149_anchored.loc[pep].values
            ax.plot(human_timepoints, y_vals, '-', color='gray', alpha=0.3, linewidth=1)

        # 149 mean
        ax.plot(human_timepoints, means_149.values, 'o-',
               color=COLORS['149']['main'], linewidth=4, markersize=12,
               label='149 Mean', zorder=5, markeredgecolor='black', markeredgewidth=2)

        # Model fit
        t_smooth = np.linspace(1, 9, 100)
        y_smooth = best_model['func'](t_smooth, *best_model['params'])
        ax.plot(t_smooth, y_smooth, '--', color='#E67E22', linewidth=2,
               alpha=0.7, label=f'{best_model_name} Fit (R²={best_model["r2"]:.3f})')

        # Reference lines
        ax.axhline(y=0, color='green', linestyle='-', linewidth=3, alpha=0.7,
                  label='Baseline (V1=NCE=0)', zorder=3)
        ax.axhline(y=1, color='red', linestyle='-', linewidth=3, alpha=0.7,
                  label='Endpoint (V9=DE=1)', zorder=3)

        # Mouse
        ax.scatter([1], [nce_mean], color=COLORS['NCE'], s=400, marker='s',
                  zorder=8, edgecolor='black', linewidth=2.5, label='NCE (Mouse)')

        # NCL (individual + mean)
        for pep in concordant_up:
            ncl_val = df_mouse_anchored.loc[pep, 'NCL']
            ncl_t = ncl_positions_dict[pep]['time']
            ax.scatter([ncl_t], [ncl_val], color=COLORS['NCL'], s=100, marker='^',
                      alpha=0.4, edgecolor='black', linewidth=0.5, zorder=6)

        ax.scatter([ncl_projected_time], [ncl_mean], color=COLORS['NCL'], s=500, marker='^',
                  zorder=8, edgecolor='black', linewidth=3,
                  label=f'NCL (t≈{ncl_projected_time:.1f})')

        # NCL projection lines
        ax.axvline(x=ncl_projected_time, color=COLORS['NCL'], linestyle='--',
                  linewidth=2, alpha=0.6, zorder=2)
        ax.axhline(y=ncl_mean, color=COLORS['NCL'], linestyle='--',
                  linewidth=2, alpha=0.6, zorder=2)

        # NCL annotation
        ax.annotate(f'NCL\n(t={ncl_projected_time:.1f})\n{ncl_mean:.2f}',
                   (ncl_projected_time, ncl_mean),
                   textcoords="offset points", xytext=(30, 30),
                   ha='left', fontsize=11, fontweight='bold',
                   bbox=dict(boxstyle='round,pad=0.5', facecolor=COLORS['NCL'],
                            alpha=0.7, edgecolor='black', linewidth=2),
                   arrowprops=dict(arrowstyle='->', lw=2, color='black'))

        # DE
        ax.scatter([9], [de_mean], color=COLORS['DE'], s=400, marker='D',
                  zorder=8, edgecolor='black', linewidth=2.5, label='DE (Mouse)')

        # Visit labels
        visit_labels = ['V1\n(NC)', 'V3\n(MCI)', 'V5\n(MCI)', 'V7\n(PDD)', 'V9\n(PDD)']
        for t, val, label in zip(human_timepoints, means_149.values, visit_labels):
            ax.annotate(label, (t, val), textcoords="offset points",
                       xytext=(0, 20), ha='center', fontsize=10, fontweight='bold',
                       bbox=dict(boxstyle='round,pad=0.3', facecolor=COLORS['149']['main'],
                                alpha=0.3, edgecolor='black'))

        ax.set_xlabel('Disease Timeline (Visit)', fontsize=13, fontweight='bold')
        ax.set_ylabel('Normalized Position (0=Baseline, 1=Endpoint)', fontsize=13, fontweight='bold')
        ax.set_title(f'Anchored Trajectory Analysis: NCL Position on 149 Timeline\n(n={len(concordant_up)} Concordant ↑ peptides)',
                    fontsize=14, fontweight='bold')
        ax.legend(fontsize=10, loc='upper left', framealpha=0.9)
        ax.grid(True, alpha=0.3)
        ax.set_xlim(0, 10)

        # Y-axis range
        y_max = max(means_149.max(), ncl_mean, de_mean) * 1.2
        y_min = min(means_149.min(), nce_mean) - 0.2
        ax.set_ylim(y_min, y_max)

        # Secondary axis
        ax2 = ax.twinx()
        ax2.set_ylim(y_min, y_max)
        ax2.set_ylabel('Progression (%)', fontsize=13, fontweight='bold')
        tick_positions = [0, 0.25, 0.5, 0.75, 1.0]
        ax2.set_yticks(tick_positions)
        ax2.set_yticklabels(['0%', '25%', '50%', '75%', '100%'])

        plt.tight_layout()
        pdf.savefig(fig, bbox_inches='tight', dpi=300)
        plt.close()

        # Individual peptide trajectories
        n_peps = len(concordant_up)
        n_cols = 3
        n_rows = int(np.ceil(n_peps / n_cols))

        fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5 * n_rows))
        if n_rows == 1:
            axes = axes.reshape(1, -1)
        axes = axes.flatten()

        for idx, pep in enumerate(concordant_up):
            ax = axes[idx]

            y_149 = df_149_anchored.loc[pep].values
            ax.plot(human_timepoints, y_149, 'o-', color=COLORS['149']['main'],
                   linewidth=2, markersize=8, label='149',
                   markeredgecolor='black', markeredgewidth=1)

            nce_val = df_mouse_anchored.loc[pep, 'NCE']
            ncl_val = df_mouse_anchored.loc[pep, 'NCL']
            de_val = df_mouse_anchored.loc[pep, 'DE']
            ncl_t = ncl_positions_dict[pep]['time']

            ax.scatter([1], [nce_val], color=COLORS['NCE'], s=150, marker='s',
                      edgecolor='black', linewidth=1.5, label='NCE', zorder=10)
            ax.scatter([ncl_t], [ncl_val], color=COLORS['NCL'], s=200, marker='^',
                      edgecolor='black', linewidth=2, label=f'NCL (t={ncl_t:.1f})', zorder=10)
            ax.scatter([9], [de_val], color=COLORS['DE'], s=150, marker='D',
                      edgecolor='black', linewidth=1.5, label='DE', zorder=10)

            ax.plot([1, ncl_t], [nce_val, ncl_val], ':', color=COLORS['Mouse']['main'],
                   linewidth=2, alpha=0.5)
            ax.plot([ncl_t, 9], [ncl_val, de_val], ':', color=COLORS['Mouse']['main'],
                   linewidth=2, alpha=0.5)

            ax.axhline(y=0, color='green', linestyle='--', alpha=0.3)
            ax.axhline(y=1, color='red', linestyle='--', alpha=0.3)
            ax.axvline(x=ncl_t, color=COLORS['NCL'], linestyle=':', alpha=0.3)

            ax.set_xlim(0, 10)
            y_max_pep = max(y_149.max(), ncl_val, de_val) * 1.2
            y_min_pep = min(y_149.min(), nce_val) - 0.2
            ax.set_ylim(y_min_pep, y_max_pep)

            ax.set_xlabel('Timeline', fontsize=9)
            ax.set_ylabel('Position (0-1)', fontsize=9)

            short_name = pep[:35] + '...' if len(pep) > 35 else pep
            ax.set_title(f'{short_name}\nNCL ≈ V{ncl_t:.1f}', fontsize=9, fontweight='bold')
            ax.legend(fontsize=7, loc='best')
            ax.grid(True, alpha=0.3)

        for idx in range(n_peps, len(axes)):
            axes[idx].axis('off')

        plt.suptitle('Individual Peptide Trajectories with NCL Positions',
                    fontsize=14, fontweight='bold')
        plt.tight_layout()
        pdf.savefig(fig, bbox_inches='tight', dpi=300)
        plt.close()

        # NCL distribution
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))

        ncl_values = [d['value'] for d in ncl_positions_dict.values()]
        ncl_times = [d['time'] for d in ncl_positions_dict.values()]

        # Histogram (0-1)
        ax = axes[0, 0]
        ax.hist(ncl_values, bins=10, color=COLORS['NCL'], alpha=0.7, edgecolor='black')
        ax.axvline(x=ncl_mean, color='darkgreen', linestyle='-', linewidth=3,
                  label=f'Mean = {ncl_mean:.3f}')
        ax.axvline(x=0, color='green', linestyle='--', linewidth=2, alpha=0.5, label='Baseline')
        ax.axvline(x=1, color='red', linestyle='--', linewidth=2, alpha=0.5, label='Endpoint')
        ax.set_xlabel('NCL Position (0-1)', fontsize=10)
        ax.set_ylabel('Count', fontsize=10)
        ax.set_title('NCL Position Distribution', fontsize=11, fontweight='bold')
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)

        # Histogram (timeline)
        ax = axes[0, 1]
        ax.hist(ncl_times, bins=10, color=COLORS['NCL'], alpha=0.7, edgecolor='black')
        ax.axvline(x=ncl_projected_time, color='darkgreen', linestyle='-', linewidth=3,
                  label=f'Mean = V{ncl_projected_time:.2f}')

        for t, label in zip([1, 3, 5, 7, 9], ['V1', 'V3', 'V5', 'V7', 'V9']):
            ax.axvline(x=t, color='gray', linestyle=':', alpha=0.5)
            ax.text(t, ax.get_ylim()[1]*0.95, label, ha='center', fontsize=9, fontweight='bold')

        ax.set_xlabel('Projected Timeline', fontsize=10)
        ax.set_ylabel('Count', fontsize=10)
        ax.set_title('NCL Timeline Distribution', fontsize=11, fontweight='bold')
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
        ax.set_xlim(0, 10)

        # Scatter
        ax = axes[1, 0]
        ax.scatter(ncl_times, ncl_values, s=150, color=COLORS['NCL'], alpha=0.7,
                  edgecolor='black', linewidth=1.5)
        ax.scatter([ncl_projected_time], [ncl_mean], s=400, color='darkgreen',
                  marker='*', edgecolor='black', linewidth=2, label='Mean', zorder=10)

        t_plot = np.linspace(1, 9, 100)
        y_plot = best_model['func'](t_plot, *best_model['params'])
        ax.plot(t_plot, y_plot, '--', color='orange', linewidth=2, alpha=0.7,
               label='149 Trajectory')

        ax.set_xlabel('Projected Timeline', fontsize=10)
        ax.set_ylabel('NCL Position (0-1)', fontsize=10)
        ax.set_title('NCL: Position vs Timeline', fontsize=11, fontweight='bold')
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
        ax.set_xlim(0, 10)

        # Summary
        ax = axes[1, 1]
        ax.axis('off')

        stage_counts = {
            'V1-V3': sum(t <= 2 for t in ncl_times),
            'V3-V5': sum(2 < t <= 4 for t in ncl_times),
            'V5-V7': sum(4 < t <= 6 for t in ncl_times),
            'V7-V9': sum(6 < t for t in ncl_times)
        }

        summary_text = f"""
    NCL Position Analysis Summary
    {'='*50}

    Concordant ↑ Peptides: {len(concordant_up)}

    NCL Position (0-1 scale):
      • Mean: {ncl_mean:.3f} ± {df_mouse_anchored['NCL'].std():.3f}
      • Range: {min(ncl_values):.3f} - {max(ncl_values):.3f}

    NCL Projected Timeline:
      • Mean: V{ncl_projected_time:.2f}
      • Range: V{min(ncl_times):.2f} - V{max(ncl_times):.2f}

    Stage Distribution:
      • V1-V3: {stage_counts['V1-V3']} peptides
      • V3-V5: {stage_counts['V3-V5']} peptides
      • V5-V7: {stage_counts['V5-V7']} peptides
      • V7-V9: {stage_counts['V7-V9']} peptides

    Model:
      • Type: {best_model_name}
      • R²: {best_model['r2']:.4f}

    Conclusion:
      NCL (10-month mouse) ≈ V{ncl_projected_time:.1f}
        """

        ax.text(0.1, 0.9, summary_text, transform=ax.transAxes, fontsize=10,
               verticalalignment='top', fontfamily='monospace',
               bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))

        plt.suptitle('NCL Position Analysis', fontsize=14, fontweight='bold')
        plt.tight_layout()
        pdf.savefig(fig, bbox_inches='tight', dpi=300)
        plt.close()

    # Save trajectory results
    with pd.ExcelWriter(f'{OUTPUT_PREFIX}_04_Trajectory_Results.xlsx', engine='openpyxl') as writer:
        df_149_anchored.to_excel(writer, sheet_name='149_Anchored')
        df_mouse_anchored.to_excel(writer, sheet_name='Mouse_Anchored')

        ncl_df = pd.DataFrame({
            'Peptide': list(ncl_positions_dict.keys()),
            'NCL_Value': [d['value'] for d in ncl_positions_dict.values()],
            'NCL_Time': [d['time'] for d in ncl_positions_dict.values()]
        })
        ncl_df.to_excel(writer, sheet_name='NCL_Positions', index=False)

        summary = pd.DataFrame({
            'Metric': ['Concordant Peptides', 'NCL Mean Position (0-1)',
                       'NCL Projected Time', 'Model Type', 'Model R²'],
            'Value': [len(concordant_up), f'{ncl_mean:.3f}',
                     f'{ncl_projected_time:.2f}', best_model_name,
                     f'{best_model["r2"]:.4f}']
        })
        summary.to_excel(writer, sheet_name='Summary', index=False)

    print(f"\n✅ Trajectory analysis complete!")
    print(f"   Saved: {OUTPUT_PREFIX}_04_Trajectory_Analysis.pdf")
    print(f"   Saved: {OUTPUT_PREFIX}_04_Trajectory_Results.xlsx")
    print(f"\n   NCL Position: {ncl_mean:.3f} (0-1 scale)")
    print(f"   NCL Timeline: V{ncl_projected_time:.2f}")

else:
    print("\nSkipping trajectory analysis (no concordant ↑ peptides)")

# ============================================================================
# Final Summary
# ============================================================================

print("\n" + "="*80)
print("ANALYSIS COMPLETE!")
print("="*80)
print(f"\nGenerated files:")
print(f"  1. {OUTPUT_PREFIX}_01_Preprocessed_Data.xlsx")
print(f"  2. {OUTPUT_PREFIX}_02_Heatmaps.pdf")
print(f"  3. {OUTPUT_PREFIX}_03_Pattern_Analysis.pdf")
print(f"  4. {OUTPUT_PREFIX}_03_Pattern_Results.xlsx")
if len(concordant_up) > 0:
    print(f"  5. {OUTPUT_PREFIX}_04_Trajectory_Analysis.pdf")
    print(f"  6. {OUTPUT_PREFIX}_04_Trajectory_Results.xlsx")

print(f"\nKey Results:")
print(f"  • Common peptides: {len(common_all_list)}")
print(f"  • Concordant peptides: {n_concordant_total} ({n_concordant_total/n_total*100:.1f}%)")
print(f"  • Pattern p-value: {p_value:.4f}")
if len(concordant_up) > 0:
    print(f"  • NCL position: V{ncl_projected_time:.2f}")

print("\n" + "="*80)